# Exercise 4 — Direct density estimation with normalizing flows

In the previous exercises we estimated density ratios directly with classifiers. Here we do a complementary exercise: train two normalizing flows,

$$
\hat p_\mathrm{sig}(x), \qquad \hat p_\mathrm{bkg}(x),
$$

where $x=(x_1,\ldots,x_5)$ are the reconstructed/smeared features. We then:

1. train one binary-mask RealNVP flow for the background and one for the signal;
2. validate the learned densities against MC projections and, for this toy problem, the analytic smeared Gaussian-mixture truth;
3. build the likelihood **directly from the densities**, without forming density ratios and without constructing a workspace;
4. fit the signal-strength parameter $\mu$ with a small likelihood;
5. use the flow-based toy density to generate pseudo-experiments and visualize the distribution of the likelihood-ratio test statistic.

### Additions in this copy

- Added, just below the second truth-based validation, a simple linear histogram of $\hat p_S(x)$ on signal events and $\hat p_B(x)$ on background events.
- Added, at the end, a flow-based toy-throwing section that samples from the trained flows and evaluates a profile-likelihood test statistic using the flow estimates of $p_S$ and $p_B$.

In [ ]:
## ============================================================================
# Google Colab setup — run me first. Safe to re-run; a no-op off Colab.
# ============================================================================
import os, sys

# --- config -----------------------------------------------------------------
REPO_URL = "https://github.com/rafaellopesdesa/nsbi-lhc-toolkit.git"  # package + tutorial helpers
BRANCH = "ml4hep_school_tutorial"
N_BKG, N_SIG = 2_000_000, 2_000_000  # Colab-sized dataset; raise for less MC noise
USE_DRIVE = True  # True -> save data/models to Google Drive so they persist
# ----------------------------------------------------------------------------

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = "/content/drive/MyDrive/Colab Notebooks/ml4hep_tifr_colab"
    else:
        ROOT = "/content"
    os.makedirs(ROOT, exist_ok=True)
    os.chdir(ROOT)

    # 1) fetch ONLY the package source + tutorial helpers (skip Git-LFS / big blobs)
    if not os.path.isdir("nsbi-lhc-toolkit"):
        os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"
        !git clone --depth 1 --filter=blob:none --sparse --branch $BRANCH $REPO_URL
        !cd nsbi-lhc-toolkit && git sparse-checkout set src workshops/ml4hep_tifr

    # 2) make `import nsbi_common_utils` work (pure-python src layout, no build step)
    src = os.path.abspath("nsbi-lhc-toolkit/src")
    if src not in sys.path:
        sys.path.insert(0, src)

    # 3) runtime deps Colab does not already ship
    !pip install -q pytorch-lightning onnx onnxruntime onnxscript iminuit mplhep

    # 4) work from the tutorial dir so utils.py / generate_distributions.py and
    #    relative paths such as ./dataframes and ./models_* resolve as in a local run
    os.chdir("nsbi-lhc-toolkit/workshops/ml4hep_tifr")

    # 5) generate the Gaussian-mixture samples if they are not there yet
    if not os.path.exists("dataframes/signal.parquet"):
        !python generate_distributions.py --n_bkg $N_BKG --n_sig $N_SIG

print("Working dir:", os.getcwd())

## Inputs expected by this notebook

This notebook expects the updated generator output:

```text
dataframes/background.parquet
dataframes/signal.parquet
```

with both truth-level columns `z1,...,z5` and reco-level columns `x1,...,x5`. The fit below uses only the reco-level `x*` variables.

In [ ]:
import math
import os
from dataclasses import dataclass
from pathlib import Path

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.special import logsumexp
from scipy.stats import multivariate_normal

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import nsbi_common_utils

from utils import (
    FEATURES,
    background_components,
    signal_components,
    smearing_parameters,
    split_train_inference,
)

FEATURES = list(FEATURES)
N_DIM = len(FEATURES)

SEED = 12345
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Features: {FEATURES}")

In [ ]:
BASE_PATH = Path("./dataframes")
FLOW_MODEL_DIR = Path("models_flows")
DENSITY_DIR = Path("saved_densities_flows")
PLOT_DIR = Path("plots_flows")

for directory in [FLOW_MODEL_DIR, DENSITY_DIR, PLOT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Must match the density-ratio training/fitting notebooks if you want disjoint
# train/inference samples across all approaches.
SPLIT_SEED = 0
TRAIN_FRACTION = 0.5

# Keep these modest for a tutorial. Increase N_EPOCHS and MAX_TRAIN_EVENTS
# for tighter closure against the analytic truth.
MAX_TRAIN_EVENTS = {
    "background": 1_000_000,
    "signal": 1_000_000,
}

N_COUPLING_LAYERS = 8
HIDDEN_FEATURES = 1024
HIDDEN_LAYERS = 4
BATCH_SIZE = 2048
N_EPOCHS = 25
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0
VALIDATION_FRACTION = 0.20
PATIENCE = 5

# If checkpoints already exist, load them instead of retraining.
LOAD_IF_AVAILABLE = False

## Load data and make the same train/inference split

The flows are trained on the training half. The likelihood fit is evaluated on the inference half. This mirrors the density-ratio notebooks and avoids using the same events to learn and evaluate the likelihood.

In [ ]:
signal = pd.read_parquet(BASE_PATH / "signal.parquet")
background = pd.read_parquet(BASE_PATH / "background.parquet")

missing = [f for f in FEATURES if f not in signal.columns or f not in background.columns]
if missing:
    raise RuntimeError(
        "Missing reco-level columns "
        f"{missing}. Regenerate the parquet files with the updated generator "
        "that writes x1,...,x5."
    )

for name, df in [("signal", signal), ("background", background)]:
    if "weight" not in df.columns:
        raise RuntimeError(f"{name} dataframe is missing a 'weight' column.")
    print(
        f"{name:10s}: {len(df):,} events, "
        f"sum weights = {df['weight'].sum():.6g}"
    )

In [ ]:
signal_train, _ = split_train_inference(
    signal, train_fraction=TRAIN_FRACTION, seed=SPLIT_SEED
)
background_train, _ = split_train_inference(
    background, train_fraction=TRAIN_FRACTION, seed=SPLIT_SEED
)

TOTAL_YIELD = {
    "signal": float(signal_train["weight"].sum()),
    "background": float(background_train["weight"].sum()),
}

# Let's use the same sample to train and eval.
# This should not be done in real life.
signal_eval = signal_train
background_eval = background_train

print(f"Flow training: {len(signal_train):,} signal + {len(background_train):,} background events")
print(f"Fit/eval: {len(signal_eval):,} signal + {len(background_eval):,} background events")
print(TOTAL_YIELD)

## A minimal RealNVP density estimator

This is a deliberately compact RealNVP implementation. Each affine-coupling layer uses a fixed binary mask, alternating which coordinates are transformed. We train by maximizing the exact log likelihood

$$
\log \hat p(x) = \log p_Z\big(f(x)\big) + \log\left|\det\frac{\partial f}{\partial x}\right|.
$$

For numerical stability, each sample is standardized before training. The final `flow_log_prob_x` helper below adds the standardization Jacobian back, so it returns a density in the original `x` coordinates.

In [ ]:
@dataclass
class Standardizer:
    mean: np.ndarray
    std: np.ndarray

    @classmethod
    def fit(cls, x):
        x = np.asarray(x, dtype=np.float32)
        mean = x.mean(axis=0).astype(np.float32)
        std = x.std(axis=0).astype(np.float32)
        std = np.where(std > 1e-6, std, 1.0).astype(np.float32)
        return cls(mean=mean, std=std)

    def transform(self, x):
        x = np.asarray(x, dtype=np.float32)
        return ((x - self.mean) / self.std).astype(np.float32)

    def inverse(self, z):
        z = np.asarray(z, dtype=np.float32)
        return (z * self.std + self.mean).astype(np.float32)

    @property
    def log_det_x_to_z_standardization(self):
        # z = (x - mean) / std, so log |dz/dx| = -sum(log std)
        return float(-np.log(self.std).sum())


class MLP(nn.Module):
    def __init__(self, n_in, n_out, hidden_features=128, hidden_layers=2):
        super().__init__()
        layers = []
        last = n_in
        for _ in range(hidden_layers):
            layers.extend([nn.Linear(last, hidden_features), nn.ReLU()])
            last = hidden_features
        layers.append(nn.Linear(last, n_out))
        self.net = nn.Sequential(*layers)

        # Start each coupling layer close to the identity map.
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, x):
        return self.net(x)


class AffineCoupling(nn.Module):
    def __init__(self, n_features, mask, hidden_features=128, hidden_layers=2, scale_clip=1.5):
        super().__init__()
        self.register_buffer("mask", torch.as_tensor(mask, dtype=torch.float32))
        self.net = MLP(
            n_features,
            2 * n_features,
            hidden_features=hidden_features,
            hidden_layers=hidden_layers,
        )
        self.scale_clip = float(scale_clip)

    def _shift_and_log_scale(self, x_masked):
        shift, log_scale = self.net(x_masked).chunk(2, dim=-1)
        inv_mask = 1.0 - self.mask
        log_scale = self.scale_clip * torch.tanh(log_scale) * inv_mask
        shift = shift * inv_mask
        return shift, log_scale

    def forward(self, x):
        """Map data space -> base space for this layer."""
        x_masked = x * self.mask
        shift, log_scale = self._shift_and_log_scale(x_masked)
        z = x_masked + (1.0 - self.mask) * (x - shift) * torch.exp(-log_scale)
        log_det = -log_scale.sum(dim=-1)
        return z, log_det

    def inverse(self, z):
        """Map base space -> data space for this layer."""
        z_masked = z * self.mask
        shift, log_scale = self._shift_and_log_scale(z_masked)
        x = z_masked + (1.0 - self.mask) * (z * torch.exp(log_scale) + shift)
        log_det = log_scale.sum(dim=-1)
        return x, log_det


class RealNVP(nn.Module):
    def __init__(
        self,
        n_features,
        n_coupling_layers=8,
        hidden_features=128,
        hidden_layers=2,
    ):
        super().__init__()
        base_mask = torch.tensor(
            [(i % 2) for i in range(n_features)], dtype=torch.float32
        )
        masks = [base_mask if i % 2 == 0 else 1.0 - base_mask for i in range(n_coupling_layers)]
        self.layers = nn.ModuleList(
            [
                AffineCoupling(
                    n_features=n_features,
                    mask=mask,
                    hidden_features=hidden_features,
                    hidden_layers=hidden_layers,
                )
                for mask in masks
            ]
        )
        self.n_features = int(n_features)

    def log_prob(self, x):
        z = x
        total_log_det = torch.zeros(x.shape[0], device=x.device)
        for layer in self.layers:
            z, log_det = layer(z)
            total_log_det = total_log_det + log_det
        base_log_prob = -0.5 * (z.pow(2) + math.log(2.0 * math.pi)).sum(dim=-1)
        return base_log_prob + total_log_det

    @torch.no_grad()
    def sample(self, n):
        z = torch.randn(n, self.n_features, device=next(self.parameters()).device)
        x = z
        for layer in reversed(self.layers):
            x, _ = layer.inverse(x)
        return x

## Training helpers

The `train_flow` helper saves checkpoints under `models_flows/`. Re-running the notebook will load existing checkpoints if `LOAD_IF_AVAILABLE = True`.

In [ ]:
def model_config():
    return {
        "n_features": N_DIM,
        "n_coupling_layers": N_COUPLING_LAYERS,
        "hidden_features": HIDDEN_FEATURES,
        "hidden_layers": HIDDEN_LAYERS,
    }


def build_flow():
    return RealNVP(**model_config()).to(device)


def checkpoint_path(sample_name):
    return FLOW_MODEL_DIR / f"realnvp_{sample_name}.pt"


def _torch_load(path):
    # PyTorch versions differ in whether torch.load exposes weights_only.
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


def save_flow(sample_name, flow, scaler):
    path = checkpoint_path(sample_name)
    torch.save(
        {
            "state_dict": flow.state_dict(),
            "scaler_mean": scaler.mean,
            "scaler_std": scaler.std,
            "features": FEATURES,
            "config": model_config(),
        },
        path,
    )
    return path


def load_flow(sample_name):
    path = checkpoint_path(sample_name)
    ckpt = _torch_load(path)
    flow = RealNVP(**ckpt["config"]).to(device)
    flow.load_state_dict(ckpt["state_dict"])
    flow.eval()
    scaler = Standardizer(
        mean=np.asarray(ckpt["scaler_mean"], dtype=np.float32),
        std=np.asarray(ckpt["scaler_std"], dtype=np.float32),
    )
    return {"flow": flow, "scaler": scaler, "path": path}


def choose_training_array(df, sample_name, seed):
    n_max = MAX_TRAIN_EVENTS.get(sample_name)
    if n_max is not None and len(df) > n_max:
        df = df.sample(n=n_max, random_state=seed).reset_index(drop=True)
    return df[FEATURES].to_numpy(dtype=np.float32)


def make_loaders(x_scaled, seed):
    x_tensor = torch.tensor(x_scaled, dtype=torch.float32)
    n_total = len(x_tensor)
    n_val = max(1, int(round(VALIDATION_FRACTION * n_total)))
    n_train = n_total - n_val

    generator = torch.Generator().manual_seed(seed)
    permutation = torch.randperm(n_total, generator=generator)
    train_idx = permutation[:n_train]
    val_idx = permutation[n_train:]

    train_ds = TensorDataset(x_tensor[train_idx])
    val_ds = TensorDataset(x_tensor[val_idx])

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        generator=generator,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
    )
    return train_loader, val_loader


def train_flow(sample_name, df_train, seed=0):
    path = checkpoint_path(sample_name)
    if LOAD_IF_AVAILABLE and path.exists():
        print(f"Loading existing {sample_name} flow from {path}")
        return load_flow(sample_name)

    x = choose_training_array(df_train, sample_name=sample_name, seed=seed)
    scaler = Standardizer.fit(x)
    x_scaled = scaler.transform(x)
    train_loader, val_loader = make_loaders(x_scaled, seed=seed)

    flow = build_flow()
    optimizer = torch.optim.AdamW(
        flow.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )

    best_val = np.inf
    best_state = None
    stale_epochs = 0

    print(f"Training {sample_name} flow on {len(x_scaled):,} events")
    for epoch in range(1, N_EPOCHS + 1):
        flow.train()
        train_losses = []
        for (batch,) in train_loader:
            batch = batch.to(device)
            loss = -flow.log_prob(batch).mean()
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(flow.parameters(), max_norm=5.0)
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        flow.eval()
        val_losses = []
        with torch.no_grad():
            for (batch,) in val_loader:
                batch = batch.to(device)
                val_losses.append(float((-flow.log_prob(batch).mean()).detach().cpu()))
        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        print(f" epoch {epoch:03d}: train NLL = {train_loss:.4f}, val NLL = {val_loss:.4f}")

        if val_loss < best_val - 1e-4:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in flow.state_dict().items()}
            stale_epochs = 0
        else:
            stale_epochs += 1
            if stale_epochs >= PATIENCE:
                print(f" early stopping after {epoch} epochs")
                break

    if best_state is not None:
        flow.load_state_dict(best_state)
    flow.eval()
    saved = save_flow(sample_name, flow, scaler)
    print(f"Saved {sample_name} flow to {saved}")
    return {"flow": flow, "scaler": scaler, "path": saved}

In [ ]:
flows = {
    "background": train_flow("background", background_train, seed=101),
    "signal": train_flow("signal", signal_train, seed=202),
}

## Density and sampling helpers

`flow_log_prob_x` returns log densities in the original `x` coordinates, not standardized coordinates.

In [ ]:
@torch.no_grad()
def flow_log_prob_x(flow_pack, x, batch_size=65_536):
    if isinstance(x, pd.DataFrame):
        x = x[FEATURES].to_numpy(dtype=np.float32)
    else:
        x = np.asarray(x, dtype=np.float32)

    flow = flow_pack["flow"]
    scaler = flow_pack["scaler"]
    flow.eval()

    chunks = []
    for start in range(0, len(x), batch_size):
        xb = scaler.transform(x[start:start + batch_size])
        xb = torch.tensor(xb, dtype=torch.float32, device=device)
        log_p_scaled = flow.log_prob(xb).detach().cpu().numpy()
        log_p_x = log_p_scaled + scaler.log_det_x_to_z_standardization
        chunks.append(log_p_x)
    if not chunks:
        return np.empty(0, dtype=float)
    return np.concatenate(chunks)


@torch.no_grad()
def flow_sample_x(flow_pack, n, batch_size=65_536):
    flow = flow_pack["flow"]
    scaler = flow_pack["scaler"]
    flow.eval()
    chunks = []
    remaining = int(n)
    while remaining > 0:
        m = min(batch_size, remaining)
        z = flow.sample(m).detach().cpu().numpy()
        chunks.append(scaler.inverse(z))
        remaining -= m
    if not chunks:
        return np.empty((0, N_DIM), dtype=np.float32)
    return np.concatenate(chunks, axis=0)

## Validation 1: MC projection closure

Generate samples from the trained flow and compare one-dimensional projections with the held-out MC sample. This is not a full high-dimensional validation, but it is an intuitive first check for students.

In [ ]:
def plot_1d_flow_closure(sample_name, df_eval, flow_pack, n_plot=50_000, n_bins=60):
    n_plot = min(n_plot, len(df_eval))
    mc = df_eval.sample(n=n_plot, random_state=SEED)[FEATURES].to_numpy(dtype=np.float32)
    gen = flow_sample_x(flow_pack, n_plot)

    fig, axes = plt.subplots(1, len(FEATURES), figsize=(4.0 * len(FEATURES), 3.2))
    if len(FEATURES) == 1:
        axes = [axes]
    for j, (ax, feat) in enumerate(zip(axes, FEATURES)):
        lo, hi = np.quantile(mc[:, j], [0.005, 0.995])
        bins = np.linspace(lo, hi, n_bins + 1)
        ax.hist(mc[:, j], bins=bins, density=True, histtype="step", lw=2, label="held-out MC")
        ax.hist(gen[:, j], bins=bins, density=True, histtype="step", lw=2, label="flow sample")
        ax.set_xlabel(feat)
        ax.set_ylabel("density")
        ax.set_title(sample_name)
    axes[0].legend(fontsize=9)
    fig.tight_layout()
    return fig

fig = plot_1d_flow_closure("background", background_eval, flows["background"])
plt.show()

fig = plot_1d_flow_closure("signal", signal_eval, flows["signal"])
plt.show()

## Validation 2: compare with the analytic smeared truth

For this toy problem we know the truth-level Gaussian mixtures and the detector response is linear-Gaussian,

$$
x = D y + \epsilon.
$$

Therefore each truth Gaussian remains a Gaussian in reconstructed space, with transformed mean and covariance. This gives an analytic density for the smeared signal and background and lets us check the flow density directly.

In [ ]:
def reco_components_from_truth_components(components):
    scale, resolution = smearing_parameters()
    scale = np.asarray(scale, dtype=float)
    resolution = np.asarray(resolution, dtype=float)
    D = np.diag(scale)
    response_cov = np.diag(resolution ** 2)

    reco_components = []
    for frac, mean_y, cov_y in components:
        mean_x = scale * np.asarray(mean_y, dtype=float)
        cov_x = D @ np.asarray(cov_y, dtype=float) @ D.T + response_cov
        reco_components.append((frac, mean_x, cov_x))
    return reco_components


def mixture_log_density(x, components):
    x = np.asarray(x, dtype=float)
    fracs = np.asarray([c[0] for c in components], dtype=float)
    fracs = fracs / fracs.sum()

    terms = []
    for f, (_, mean, cov) in zip(fracs, components):
        terms.append(
            np.log(f) + multivariate_normal(mean=mean, cov=cov, allow_singular=False).logpdf(x)
        )
    return logsumexp(np.vstack(terms), axis=0)


TRUTH_RECO_COMPONENTS = {
    "background": reco_components_from_truth_components(background_components()),
    "signal": reco_components_from_truth_components(signal_components()),
}

In [ ]:
def compare_log_density_to_truth(sample_name, df_eval, flow_pack, n_points=25_000):
    n_points = min(n_points, len(df_eval))
    x = df_eval.sample(n=n_points, random_state=SEED)[FEATURES].to_numpy(dtype=np.float32)
    log_p_flow = flow_log_prob_x(flow_pack, x)
    log_p_truth = mixture_log_density(x, TRUTH_RECO_COMPONENTS[sample_name])
    delta = log_p_flow - log_p_truth

    corr = np.corrcoef(log_p_truth, log_p_flow)[0, 1]
    rmse = np.sqrt(np.mean(delta ** 2))
    bias = np.mean(delta)
    print(f"{sample_name}: corr(log p) = {corr:.4f}, RMSE = {rmse:.4f}, bias = {bias:.4f}")

    fig, ax = plt.subplots(figsize=(5.5, 5.0))
    ax.scatter(log_p_truth, log_p_flow, s=4, alpha=0.25)
    lo = min(log_p_truth.min(), log_p_flow.min())
    hi = max(log_p_truth.max(), log_p_flow.max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_xlabel("truth log density")
    ax.set_ylabel("flow log density")
    ax.set_title(sample_name)
    fig.tight_layout()
    return fig

fig = compare_log_density_to_truth("background", background_eval, flows["background"])
plt.show()

fig = compare_log_density_to_truth("signal", signal_eval, flows["signal"])
plt.show()

## Flow density histograms on native samples

The plots below use the flow density itself, not its logarithm. The left panel shows $\hat p_S(x)$ evaluated on events from the signal sample. The right panel shows $\hat p_B(x)$ evaluated on events from the background sample.

In [ ]:
def plot_native_flow_density_histograms(n_points=100_000, n_bins=80):
    """Simple linear-scale histograms of the flow density on native samples."""
    n_sig = min(n_points, len(signal_eval))
    n_bkg = min(n_points, len(background_eval))

    x_sig = signal_eval.sample(n=n_sig, random_state=SEED)[FEATURES].to_numpy(dtype=np.float32)
    x_bkg = background_eval.sample(n=n_bkg, random_state=SEED + 1)[FEATURES].to_numpy(dtype=np.float32)

    log_p_s_on_signal = flow_log_prob_x(flows["signal"], x_sig)
    log_p_b_on_background = flow_log_prob_x(flows["background"], x_bkg)

    # Convert to p, not log p. Clip only to avoid floating-point under/overflow warnings.
    p_s_on_signal = np.exp(np.clip(log_p_s_on_signal, -745.0, 700.0))
    p_b_on_background = np.exp(np.clip(log_p_b_on_background, -745.0, 700.0))

    fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.0))

    axes[0].hist(p_s_on_signal, bins=n_bins, histtype="step", lw=2)
    axes[0].set_xlabel(r"$\hat p_S(x)$ on signal events")
    axes[0].set_ylabel("events")
    axes[0].set_title("Flow density for signal sample")

    axes[1].hist(p_b_on_background, bins=n_bins, histtype="step", lw=2)
    axes[1].set_xlabel(r"$\hat p_B(x)$ on background events")
    axes[1].set_ylabel("events")
    axes[1].set_title("Flow density for background sample")

    for ax in axes:
        ax.ticklabel_format(axis="x", style="sci", scilimits=(0, 0))

    fig.tight_layout()
    fig.savefig(PLOT_DIR / "flow_density_native_samples.png", dpi=150)
    return fig, {
        "p_s_on_signal": p_s_on_signal,
        "p_b_on_background": p_b_on_background,
    }

fig, native_density_samples = plot_native_flow_density_histograms()
p_s_on_signal = native_density_samples["p_s_on_signal"]
p_b_on_background = native_density_samples["p_b_on_background"]
plt.show()

## Flow-based extended likelihood

For a toy dataset $\{x_i\}$, use the fitted flow densities directly in the extended likelihood

$$
\mathcal{L}(\mu) = e^{-(\mu S + B)} \prod_i \left[\mu S\,\hat p_S(x_i) + B\,\hat p_B(x_i)\right],
$$

where $S$ and $B$ are the expected signal and background yields and $\mu$ is the signal-strength parameter. Constants independent of $\mu$ are omitted in the negative log likelihood.

In [ ]:
def flow_log_ps_pb(x, batch_size=65_536):
    """Return log p_S(x) and log p_B(x) using the trained flows."""
    x = np.asarray(x, dtype=np.float32)
    log_ps = flow_log_prob_x(flows["signal"], x, batch_size=batch_size)
    log_pb = flow_log_prob_x(flows["background"], x, batch_size=batch_size)
    return log_ps, log_pb


def _log_yield_times_density(expected_yield, log_p):
    if expected_yield <= 0:
        return np.full_like(log_p, -np.inf, dtype=float)
    return np.log(float(expected_yield)) + log_p


def flow_extended_nll_mu(mu, log_ps, log_pb, expected_s, expected_b):
    """Extended negative log likelihood, up to additive constants independent of mu."""
    mu = float(mu)
    if mu < 0:
        return np.inf

    expected_total = mu * expected_s + expected_b
    log_signal_intensity = _log_yield_times_density(mu * expected_s, log_ps)
    log_background_intensity = _log_yield_times_density(expected_b, log_pb)
    log_event_intensity = np.logaddexp(log_signal_intensity, log_background_intensity)
    return expected_total - float(np.sum(log_event_intensity))


def fit_mu_hat_from_flow_densities(log_ps, log_pb, expected_s, expected_b, mu_max=5.0):
    """Profile the one-dimensional signal strength with mu constrained to [0, mu_max]."""
    objective = lambda mu: flow_extended_nll_mu(mu, log_ps, log_pb, expected_s, expected_b)
    result = minimize_scalar(objective, bounds=(0.0, float(mu_max)), method="bounded", options={"xatol": 1e-3})

    candidates = [
        (0.0, objective(0.0)),
        (float(mu_max), objective(mu_max)),
    ]
    if result.success:
        candidates.append((float(result.x), float(result.fun)))

    mu_hat, nll_hat = min(candidates, key=lambda item: item[1])
    return mu_hat, nll_hat


def flow_q0_test_statistic(x, expected_s, expected_b, mu_max=5.0):
    """Discovery-style q0 = -2 log lambda(0), using flow p_S and p_B."""
    x = np.asarray(x, dtype=np.float32)
    if len(x) == 0:
        return 0.0, 0.0

    log_ps, log_pb = flow_log_ps_pb(x)
    mu_hat, nll_hat = fit_mu_hat_from_flow_densities(
        log_ps, log_pb, expected_s=expected_s, expected_b=expected_b, mu_max=mu_max
    )
    nll_mu0 = flow_extended_nll_mu(0.0, log_ps, log_pb, expected_s=expected_s, expected_b=expected_b)

    # The constrained q0 is set to zero when the best fit is at mu_hat = 0.
    q0 = max(0.0, 2.0 * (nll_mu0 - nll_hat))
    if mu_hat <= 1e-6:
        q0 = 0.0
    return q0, mu_hat

## Throw toys from the flows and plot the expected test-statistic distribution

The helper below generates pseudo-experiments by drawing Poisson event counts and sampling events from the trained signal and background flows. The same trained flows provide $\hat p_S$ and $\hat p_B$ in the extended likelihood used to compute $q_0$.

For runtime, the default uses a scaled luminosity. Set `TOY_LUMI_SCALE = 1.0` for the nominal yields and increase `N_TOYS` for smoother expected distributions.

In [ ]:
def flow_toy_event_sample(mu_true, expected_s, expected_b, rng):
    """Generate one extended toy sample from the trained flows."""
    n_s = int(rng.poisson(max(mu_true * expected_s, 0.0)))
    n_b = int(rng.poisson(max(expected_b, 0.0)))

    pieces = []
    if n_s > 0:
        pieces.append(flow_sample_x(flows["signal"], n_s))
    if n_b > 0:
        pieces.append(flow_sample_x(flows["background"], n_b))

    if pieces:
        x = np.concatenate(pieces, axis=0).astype(np.float32)
        # Shuffle so the likelihood sees an unordered dataset.
        rng.shuffle(x, axis=0)
    else:
        x = np.empty((0, N_DIM), dtype=np.float32)
    return x, {"n_signal": n_s, "n_background": n_b}


def throw_flow_toys_q0(
    n_toys=100,
    mu_true=0.0,
    expected_s=None,
    expected_b=None,
    toy_lumi_scale=1e-3,
    mu_max=5.0,
    seed=SEED + 999,
):
    """Throw toys and compute q0 for each one using the flow-based likelihood."""
    if expected_s is None:
        expected_s = TOTAL_YIELD["signal"] * toy_lumi_scale
    if expected_b is None:
        expected_b = TOTAL_YIELD["background"] * toy_lumi_scale

    rng = np.random.default_rng(seed)
    q0_values = []
    mu_hats = []
    counts = []

    for i_toy in range(int(n_toys)):
        x_toy, count_info = flow_toy_event_sample(mu_true, expected_s, expected_b, rng)
        q0, mu_hat = flow_q0_test_statistic(
            x_toy, expected_s=expected_s, expected_b=expected_b, mu_max=mu_max
        )
        q0_values.append(q0)
        mu_hats.append(mu_hat)
        counts.append(count_info)

        if (i_toy + 1) % max(1, n_toys // 5) == 0:
            print(f"finished {i_toy + 1}/{n_toys} toys")

    return {
        "q0": np.asarray(q0_values, dtype=float),
        "mu_hat": np.asarray(mu_hats, dtype=float),
        "counts": counts,
        "expected_s": float(expected_s),
        "expected_b": float(expected_b),
        "mu_true": float(mu_true),
        "toy_lumi_scale": float(toy_lumi_scale),
    }


# Runtime-friendly defaults for the tutorial. For nominal yields, set TOY_LUMI_SCALE = 1.0.
N_TOYS = 100
TOY_MU_TRUE = 0.0       # background-only toys for the discovery-style q0 distribution
TOY_LUMI_SCALE = 1e-3   # yields become S~1.1 and B~1000 for the default generated sample
MU_MAX = 5.0

toy_results = throw_flow_toys_q0(
    n_toys=N_TOYS,
    mu_true=TOY_MU_TRUE,
    toy_lumi_scale=TOY_LUMI_SCALE,
    mu_max=MU_MAX,
)

q0_values = toy_results["q0"]
mu_hats = toy_results["mu_hat"]

if np.allclose(q0_values, q0_values[0]):
    center = float(q0_values[0])
    bins = np.linspace(center - 0.5, center + 0.5, 31)
else:
    bins = 40

fig, ax = plt.subplots(figsize=(6.5, 4.3))
ax.hist(q0_values, bins=bins, histtype="step", lw=2, density=True)
ax.axvline(np.median(q0_values), ls="--", lw=1, label=f"median = {np.median(q0_values):.3g}")
ax.set_xlabel(r"$q_0 = -2\log\lambda(0)$")
ax.set_ylabel("toy density")
ax.set_title(
    "Flow-based expected test-statistic distribution\n"
    rf"$\mu_{{true}}={TOY_MU_TRUE:g}$, $E[S]={toy_results['expected_s']:.3g}$, "
    rf"$E[B]={toy_results['expected_b']:.3g}$, {N_TOYS} toys"
)
ax.legend()
fig.tight_layout()
fig.savefig(PLOT_DIR / "flow_based_q0_toy_distribution.png", dpi=150)
plt.show()

lo, hi = np.quantile(q0_values, [0.16, 0.84])
mean_counts_s = np.mean([c["n_signal"] for c in toy_results["counts"]])
mean_counts_b = np.mean([c["n_background"] for c in toy_results["counts"]])
print(
    f"Toy setup: mu_true={toy_results['mu_true']:.3g}, "
    f"E[S]={toy_results['expected_s']:.6g}, E[B]={toy_results['expected_b']:.6g}, "
    f"toy_lumi_scale={toy_results['toy_lumi_scale']:.3g}"
)
print(f"Average generated counts: signal={mean_counts_s:.3g}, background={mean_counts_b:.3g}")
print(f"q0 median = {np.median(q0_values):.6g}; central 68% interval = [{lo:.6g}, {hi:.6g}]")
print(f"mu_hat median = {np.median(mu_hats):.6g}")